# NCP-Wired CfC Phase 1 - Google Colab

Bu notebook, NCP-wired CfC mimarisini Colab'da sifirdan kurar, BC + DAgger ile egitir ve 22 custom + 2 dynamic haritada degerlendirir.

Kullanim: `Runtime > Change runtime type > GPU` sec, sonra hucreleri ustten alta calistir. `RESUME_CHECKPOINT` yok; bu akis sifirdan egitim icindir.

## 1. GPU kontrolu

In [ ]:
!nvidia-smi

## 2. Repoyu klonla ve kur

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/heimdilon/mujoco_lnn_navigation.git"
REPO_DIR = Path("/content/mujoco_lnn_navigation")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/mujoco_lnn_navigation
!git checkout master
!git pull --ff-only

In [ ]:
%env MUJOCO_GL=egl
!apt-get update -qq
!apt-get install -y -qq libegl1 libgl1
!python -m pip install -U pip
!python -m pip install -e .
!python -m pip install matplotlib

## 3. Ortami dogrula

In [ ]:
import importlib.metadata as metadata
import torch

print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
print("cuda_device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("ncps package", metadata.version("ncps"))
from ncps.wirings import AutoNCP
print("AutoNCP", AutoNCP)

In [ ]:
for cfg in [
    "configs/train/bc_ncp_cfc_dynamic_maps.yaml",
    "configs/train/bc_ncp_cfc_s030_dynamic_maps.yaml",
    "configs/train/bc_ncp_cfc_u64_dynamic_maps.yaml",
    "configs/train/bc_ncp_cfc_u64_o24_s030_dynamic_maps.yaml",
]:
    print("\nSMOKE", cfg)
    !python tools/ncp_smoke_test.py --train-config {cfg}

## 4. NCP Phase 1 egitimi

In [ ]:
import torch

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True

EXPERIMENT = "s030"  # secenekler: base, s030, u64, u64_o24_s030
BATCH_SEQUENCES = 8  # RTX 6000 icin 8 guvenli; OOM yoksa 12 veya 16 denenebilir
LOG_INTERVAL = 10

EXPERIMENTS = {
    "base": {
        "run_name": "ncp_cfc_phase1",
        "train_config": "configs/train/bc_ncp_cfc_dynamic_maps.yaml",
        "note": "48 units, output 16, sparsity 0.5",
    },
    "s030": {
        "run_name": "ncp_cfc_s030_phase1",
        "train_config": "configs/train/bc_ncp_cfc_s030_dynamic_maps.yaml",
        "note": "Ablation A: only sparsity 0.5 -> 0.3",
    },
    "u64": {
        "run_name": "ncp_cfc_u64_phase1",
        "train_config": "configs/train/bc_ncp_cfc_u64_dynamic_maps.yaml",
        "note": "Ablation B: only units 48 -> 64",
    },
    "u64_o24_s030": {
        "run_name": "ncp_cfc_u64_o24_s030_phase1",
        "train_config": "configs/train/bc_ncp_cfc_u64_o24_s030_dynamic_maps.yaml",
        "note": "Ablation C: units 64, output 24, sparsity 0.3",
    },
}

selected = EXPERIMENTS[EXPERIMENT]
RUN_NAME = selected["run_name"]
SPLIT_CONFIG = "configs/splits/custom22_dynamic_seed25462877008.yaml"
TRAIN_CONFIG = selected["train_config"]
DAGGER_ITERATIONS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("experiment", EXPERIMENT)
print("note", selected["note"])
print("run", RUN_NAME)
print("train_config", TRAIN_CONFIG)
print("batch_sequences", BATCH_SEQUENCES)
print("log_interval", LOG_INTERVAL)
print("device", DEVICE)

In [ ]:
import shlex
import subprocess
import sys

train_cmd = [
    sys.executable,
    "-u",
    "scripts/train_bc.py",
    "--split-config",
    SPLIT_CONFIG,
    "--train-config",
    TRAIN_CONFIG,
    "--run-name",
    RUN_NAME,
    "--dagger-iterations",
    str(DAGGER_ITERATIONS),
    "--save-interval",
    "50",
    "--batch-sequences",
    str(BATCH_SEQUENCES),
    "--bucket-by-length",
    "--log-interval",
    str(LOG_INTERVAL),
    "--no-final-eval",
    "--device",
    DEVICE,
]
print(" ".join(shlex.quote(part) for part in train_cmd))
subprocess.run(train_cmd, check=True)

## 5. 24 harita degerlendirme

In [ ]:
import yaml

EVAL_RUN_NAME = f"{RUN_NAME}_eval_all24"
EVAL_EPISODES = 4
MAKE_VISUALS = True

with open(SPLIT_CONFIG, "r", encoding="utf-8") as handle:
    split = yaml.safe_load(handle)

map_paths = list(split["train_maps"]) + list(split["holdout_maps"])
print(len(map_paths), "maps")

In [ ]:
eval_cmd = [
    sys.executable,
    "scripts/batch_evaluate.py",
    "--map-configs",
    *map_paths,
    "--checkpoint",
    f"results/{RUN_NAME}/latest.pt",
    "--run-name",
    EVAL_RUN_NAME,
    "--episodes",
    str(EVAL_EPISODES),
    "--max-steps",
    "900",
    "--goal-observation-max",
    "10.0",
    "--device",
    "cpu",
]
if not MAKE_VISUALS:
    eval_cmd.extend(["--no-gif", "--no-png"])

print(" ".join(shlex.quote(part) for part in eval_cmd))
subprocess.run(eval_cmd, check=True)

In [ ]:
import pandas as pd

summary_path = Path("results") / EVAL_RUN_NAME / "summary.csv"
summary = pd.read_csv(summary_path)
display(summary[["map", "success_rate", "collision_rate", "timeout_rate", "mean_steps"]])

success_maps = int((summary["success_rate"] > 0.0).sum())
print(f"success maps: {success_maps}/{len(summary)}")
print("summary:", summary_path)

## 6. Sonuclari indir

In [ ]:
from google.colab import files

zip_name = f"{RUN_NAME}_artifacts.zip"
!zip -r -q {zip_name} results/{RUN_NAME} results/{EVAL_RUN_NAME}
files.download(zip_name)